# 文学研究のためのメタデータ実習

**2026年10月16日 技術講習会**

Open Libraryの書誌メタデータを使って、作品の検索から出版・版展開の分析まで体験します。

## はじめに

最初は **「ランタイム」→「すべてのセルを実行」** を選んでください。
初期設定では James Joyce の *Ulysses* を調べます。

### 自分の作品を調べるには

1. **A** の `SEARCH_TITLE` に作品名を入力し、そのコードセルを実行する。
2. **B** の `SEARCH_RESULT_NUMBER` で検索結果の番号を選び、実行する。Work IDを直接指定しても構いません。
3. **B-1** を実行すると、取得したEditionの出版年代・言語を確認できる。
4. **B-2** を実行すると、固定データにある作品について、1880〜2026年などの長期的な英語Editionの分布を調べられる。
5. **C** では、固定データに対応するWorkのWikidata・OpenAlex情報を調べられる。

**作品を変更した場合は、Aから順番に必要なセルを再実行してください。**

## データの違いに注意

- **A・B・B-1：** Open LibraryのライブAPI。B-1は取得できたEditionの範囲内の分析であり、常に全件とは限りません。
- **B-2・C：** 2026年2月28日版の固定ダンプを基礎とする、34,789件のOL Workレコードの分析。
- 固定集合は英語Editionの存在を条件に作成されたもので、英語原作の小説だけを厳密に選別したものではありません。
- Edition数は書誌レコード数です。出版部数、販売部数、読者数、作品の評価を直接表すものではありません。
- OpenAlexの候補数は暫定値であり、検証済みの研究論文数ではありません。


## A. Open Library全体で作品を検索する
英文学・日本文学など、分野を問わず調べられます。
作品名や著者名を書き換えてセルを再実行してください。

In [ ]:
import json
import pandas as pd
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from IPython.display import display

def ol_api(path, params=None):
    url = "https://openlibrary.org" + path
    if params:
        url += "?" + urlencode(params)
    req = Request(
        url,
        headers={"User-Agent": "LiteraryMetadataWorkshop2026/0.1"}
    )
    with urlopen(req, timeout=25) as response:
        return json.load(response)

SEARCH_TITLE = "Ulysses" # @param {type:"string"}
SEARCH_AUTHOR = "" # @param {type:"string"}

if not SEARCH_TITLE.strip() and not SEARCH_AUTHOR.strip():
    raise ValueError("作品名または著者名を入力してください。")

params = {
    "limit": 10,
    "fields": "key,title,author_name,first_publish_year,edition_count",
}
if SEARCH_TITLE.strip():
    params["title"] = SEARCH_TITLE.strip()
if SEARCH_AUTHOR.strip():
    params["author"] = SEARCH_AUTHOR.strip()

results = ol_api("/search.json", params)
ol_docs = results.get("docs", [])

ol_results = pd.DataFrame([
    {
        "Work ID": d.get("key", "").replace("/works/", ""),
        "作品名": d.get("title", ""),
        "著者": " / ".join(d.get("author_name") or []),
        "OL初刊年": d.get("first_publish_year"),
        "登録Edition数": d.get("edition_count"),
    }
    for d in ol_docs
])

print("Open Library全体の検索結果")
print("該当件数（API報告値）:",
      results.get("numFound", results.get("num_found")))
display(ol_results)

print("注意：OL初刊年は、歴史上の原作初刊年と一致するとは限りません。")

## B. 選んだ作品のEditionを確認する

Aの検索結果から番号を指定してWorkを選びます。
番号は1から始まります。

Work IDを直接指定することもできます。

選んだWorkは、この後の出版年比較と
Wikidata・OpenAlexの確認に引き継がれます。

取得件数に上限があるため、APIの一覧が
全Editionを網羅しているとは限りません。


In [ ]:
import re

# 空欄なら直前の検索結果の先頭Workを選択
SELECT_WORK_ID = "" # @param {type:"string"}
SEARCH_RESULT_NUMBER = 1 # @param {type:"integer", min:1, max:10}
MAX_EDITIONS = 100 # @param {type:"integer", min:10, max:200}

if SELECT_WORK_ID.strip():
    wid = SELECT_WORK_ID.strip().replace("/works/", "")
elif not ol_results.empty:
    index = int(SEARCH_RESULT_NUMBER) - 1
    if not 0 <= index < len(ol_results):
        raise ValueError("検索結果の番号を確認してください。")
    wid = str(ol_results.iloc[index]["Work ID"])
else:
    raise ValueError("検索結果がありません。Work IDを指定してください。")

if not re.fullmatch(r"OL[0-9]+W", wid):
    raise ValueError("Work IDの形式を確認してください: " + wid)

limit = max(1, min(int(MAX_EDITIONS), 200))

response = ol_api(
    f"/works/{wid}/editions.json",
    {"limit": limit, "offset": 0},
)

def edition_language_codes(record):
    return [
        v["key"].rsplit("/", 1)[-1]
        for v in (record.get("languages") or [])
        if isinstance(v, dict) and isinstance(v.get("key"), str)
    ]

edition_rows = []
for item in response.get("entries", []):
    edition_rows.append({
        "Edition ID": item.get("key", "").replace("/books/", ""),
        "タイトル": item.get("title", ""),
        "刊行年・日付": item.get("publish_date", ""),
        "言語コード": ", ".join(edition_language_codes(item)),
        "出版社": ", ".join(item.get("publishers") or []),
    })

edition_view = pd.DataFrame(edition_rows)
total = response.get("size")
n = len(edition_rows)

print("Work ID:", wid)
print("APIが報告する登録Edition総数:", total)
print("今回取得したEdition数:", n)

if isinstance(total, int) and n < total:
    print("注意：一部のEditionのみ取得しています。"
          "以下の結果は全Editionの集計ではありません。")

display(edition_view.head(30))

if not edition_view.empty:
    print("取得したEditionに付いている言語タグの例:")
    display(
        edition_view["言語コード"]
        .replace("", "未記録")
        .value_counts()
        .head(15)
        .to_frame("取得範囲内のレコード数")
    )

print("この表は検索時点のAPIデータです。固定ダンプの集計とは異なります。")

## B-1. 取得したEditionの出版年代を見る

直前に選択したWorkについて、APIから取得したEditionを年代別に集計します。

言語は「すべて」「英語」「日本語」から選択できます。
表示期間も変更できます。

**重要：** ここでは、直前のセルで取得できたEditionだけを集計します。
APIの登録総数より取得件数が少ない場合、作品全体の年代分布ではありません。

固定データに含まれる作品については、次のB-2でより広い
スナップショット集計を確認できます。


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

API_LANGUAGE = "すべて" # @param ["すべて", "英語 (eng)", "日本語 (jpn)"]
API_PERIOD = "全記録（2026年まで）" # @param ["全記録（2026年まで）", "1880〜1950年", "1951〜2026年"]

language_map = {
    "すべて": None,
    "英語 (eng)": "eng",
    "日本語 (jpn)": "jpn",
}
period_map = {
    "全記録（2026年まで）": (1500, 2026),
    "1880〜1950年": (1880, 1950),
    "1951〜2026年": (1951, 2026),
}

language_code = language_map[API_LANGUAGE]
year_min, year_max = period_map[API_PERIOD]

api_edition_sample = edition_view.copy()

# Open Libraryのpublish_dateは一定の書式とは限らない。
# 抽出できた4桁の年のみ利用し、推測で補完しない。
year_pattern = r"(?<!\d)(1[5-9]\d{2}|20\d{2})(?!\d)"

def extract_year(value):
    match = re.search(year_pattern, str(value))
    return int(match.group(1)) if match else None

api_edition_sample["parsed_year"] = (
    api_edition_sample["刊行年・日付"]
    .apply(extract_year)
)

def has_language(codes, target):
    if target is None:
        return True
    actual = {s.strip() for s in str(codes).split(",") if s.strip()}
    return target in actual

selected_api_editions = api_edition_sample[
    api_edition_sample["言語コード"].apply(
        lambda x: has_language(x, language_code)
    )
].copy()

api_plot_rows = selected_api_editions[
    selected_api_editions["parsed_year"].between(
        year_min, year_max
    )
].copy()

api_plot_rows["decade"] = (
    api_plot_rows["parsed_year"] // 10 * 10
).astype(int)

api_decade_counts = (
    api_plot_rows.groupby("decade")
    .size()
    .sort_index()
)

reported_total = response.get("size")
fetched_total = len(edition_view)

print("=== Open Library API：取得範囲内のEdition ===")
print("Work ID:", wid)
print("APIが報告するEdition総数:", reported_total)
print("実際に取得した件数:", fetched_total)
print("言語:", API_LANGUAGE)
print("表示期間:", API_PERIOD)
print("選択言語に該当する取得Edition:", len(selected_api_editions))
print("表示期間内・刊行年判読可能:", len(api_plot_rows))
print(
    "選択言語で刊行年を判読できないEdition:",
    int(selected_api_editions["parsed_year"].isna().sum())
)

if not isinstance(reported_total, int) or fetched_total < reported_total:
    print(
        "【部分集計】APIのEdition全件を取得していません。"
        "このグラフから作品全体の年代分布を推定しないでください。"
    )
else:
    print("【取得状況】API報告件数と取得件数が一致しています。")

if api_decade_counts.empty:
    print("指定した言語・年代に該当する取得Editionがありません。")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    api_decade_counts.plot.bar(ax=ax)

    ax.set_title("Edition records in the fetched API response")
    ax.set_xlabel("Recorded publication decade")
    ax.set_ylabel("Edition records")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

display(
    selected_api_editions[[
        "Edition ID",
        "タイトル",
        "刊行年・日付",
        "言語コード",
        "出版社",
    ]].head(20)
)

print(
    "注意：Edition数は印刷部数・読者数ではありません。"
    "言語タグ未記録の版は、個別言語の集計に含まれません。"
)


## B-2. 英語Editionの長期的な出版状況を見る

2026年2月28日のOpen Library固定ダンプを使用します。

出版年の記録を、1880〜1950年、その後の1951〜2026年、
または1880〜2026年の全期間で表示できます。

直前に検索・選択したWorkを引き継ぎます。
別のWork IDを追加すれば、最大5件まで比較できます。

**注意事項**

- 英語Editionであることは、原作言語が英語である証拠ではありません。
- Editionレコード数は出版部数・読者数・再評価の直接的な指標ではありません。
- 同じ文学作品が複数のOpen Library Workに分かれている可能性があります。
- 2026年の出版年は途中までのデータであり、将来刊行予定の記録もあり得ます。
- 34,789件に含まれないWorkは、この固定データでは比較できません。


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ENGLISH_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "haruka0221/literary-metadata-workshop-2026/"
    "main/data/workshop_english_editions_v1.csv"
)

HISTORY_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "haruka0221/literary-metadata-workshop-2026/"
    "main/data/work_english_edition_by_year_v1.csv"
)

try:
    english_df = pd.read_csv(
        ENGLISH_DATA_URL, dtype=str,
        keep_default_na=False, encoding="utf-8-sig"
    )
    history_df = pd.read_csv(
        HISTORY_DATA_URL, dtype=str,
        keep_default_na=False, encoding="utf-8-sig"
    )
except Exception as exc:
    raise RuntimeError(
        "英語Editionデータを取得できません。"
        "公開状況や接続を確認してください。"
    ) from exc

assert len(english_df) == 34789
assert english_df["ol_work_id"].is_unique
assert len(history_df) == 129464

assert not history_df.duplicated(
    ["ol_work_id", "edition_publish_year"]
).any()

history_df["count"] = pd.to_numeric(
    history_df["english_edition_records"],
    errors="raise"
).astype(int)

history_df["year"] = pd.to_numeric(
    history_df["edition_publish_year"].replace(
        "UNKNOWN", None
    ),
    errors="raise"
)

assert (history_df["count"] > 0).all()
assert history_df["count"].sum() == 340860

summary = english_df.set_index("ol_work_id", drop=False)

# Workごとの全集計が一致することも確認
actual_totals = history_df.groupby(
    "ol_work_id"
)["count"].sum()

expected_totals = pd.to_numeric(
    summary["english_edition_records_total"]
)

assert (
    actual_totals.reindex(
        summary.index, fill_value=0
    ).to_numpy() == expected_totals.to_numpy()
).all()

# 以下の変数はGoogle Colabのフォームから選択できます
PERIOD = "1880–2026" # @param ["1880–2026", "1880–1950", "1951–2026"]
OTHER_WORK_IDS = "" # @param {type:"string"}

ranges = {
    "1880–2026": (1880, 2026),
    "1880–1950": (1880, 1950),
    "1951–2026": (1951, 2026),
}

start, end = ranges[PERIOD]

requested = [wid] + [
    item.strip().replace("/works/", "")
    for item in OTHER_WORK_IDS.split(",")
    if item.strip()
]
requested = list(dict.fromkeys(requested))[:5]

selected = []
missing = []

for work_id in requested:
    if not re.fullmatch(r"OL[0-9]+W", work_id):
        print("不正なWork ID:", work_id)
    elif work_id not in summary.index:
        missing.append(work_id)
    else:
        selected.append(work_id)

if missing:
    print("固定データの対象外:", ", ".join(missing))
    print("対象外の作品も、前のAPI検索では調べられます。")

if selected:
    # 全体の集計表
    results = []

    # グラフの横軸：選択期間内の10年単位
    decade_starts = list(
        range((start // 10) * 10, end + 1, 10)
    )
    labels = [
        f"{max(d, start)}–{min(d + 9, end)}"
        for d in decade_starts
    ]

    fig, ax = plt.subplots(figsize=(11, 4.8))

    for work_id in selected:
        records = history_df[
            history_df["ol_work_id"] == work_id
        ]

        in_range = records[
            records["year"].between(start, end)
        ].copy()

        by_decade = in_range.groupby(
            (in_range["year"] // 10 * 10).astype(int)
        )["count"].sum()

        values = [
            int(by_decade.get(d, 0))
            for d in decade_starts
        ]

        meta = summary.loc[work_id]

        unknown = int(records.loc[
            records["year"].isna(), "count"
        ].sum())

        after_snapshot = int(records.loc[
            records["year"] > 2026, "count"
        ].sum())

        results.append({
            "Work ID": work_id,
            "作品名": meta["ol_title"],
            "著者": meta["ol_author_name"],
            f"{PERIOD}の英語Edition": sum(values),
            "全期間の英語Edition": int(
                meta["english_edition_records_total"]
            ),
            "刊行年不明": unknown,
            "2027年以降の刊行年": after_snapshot,
        })

        ax.plot(
            labels, values, marker="o",
            label=f'{meta["ol_title"][:24]} ({work_id})'
        )

    print("=== 選択した作品の英語Edition ===")
    display(pd.DataFrame(results))

    ax.set_title("English-language Edition records by publication period")
    ax.set_xlabel("Recorded publication years")
    ax.set_ylabel("Edition records")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.show()

    print("各区間の幅は境界で異なる場合があります。")
    print("刊行年不明・1880年以前・2027年以降はグラフに含めません。")
else:
    print("固定データで比較可能な作品がありません。")

print(
    "データ時点：Open Library Editions "
    "2026-02-28 snapshot"
)
print(
    "書誌レコード数は実際の出版回数や読者数を"
    "直接表すものではありません。"
)

## C. 研究用固定データでさらに分析する
ここからは2026年2月28日版を基礎とする34,789件の固定集合です。
Open Library全体検索とは対象範囲とデータ時点が異なります。
Wikidata・OpenAlexの数値は、同定・計測状態に注意して読みます。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = "haruka0221/literary-metadata-workshop-2026"
REF = "main"
URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/workshop_search_v1.csv"

try:
    df = pd.read_csv(URL, dtype=str, keep_default_na=False, encoding="utf-8-sig")
except Exception as exc:
    raise RuntimeError("講習会用CSVがまだ公開されていないか、取得できません: " + URL) from exc

FLAG_LABELS = {
    "PROVISIONAL": "暫定（作品ID接続済み）",
    "PROVISIONAL_UNRESOLVED": "暫定・作品同定未確定",
    "NOT_AVAILABLE": "候補数を表示できない",
    "IDENTITY_HOLD": "作品同定保留",
}

required = {
    "ol_work_id", "ol_title", "ol_author_name",
    "ol_first_publish_year", "ol_edition_count",
    "openalex_display_candidate_count", "openalex_quality_flag",
    "openalex_count_unit_id", "wikidata_decision",
}
assert required <= set(df.columns), "CSVの列が想定と異なります"
assert len(df) == 34789, "CSVの行数が想定と異なります"

print(f"読み込み成功：{len(df):,} OLレコード")
display(df.head(3))


## C-1. 選択した作品を研究用データに接続する

A・Bで選んだWork IDを使い、
固定された34,789件のデータを調べます。

対応するレコードがあれば、
Wikidataの照合結果とOpenAlexの暫定候補数を表示します。

固定集合に含まれないWorkでも、
A・BのOpen Library検索は使用できます。

このセクションだけで別作品を調べたい場合は、
タイトル・著者の入力欄も利用できます。


In [ ]:
# 通常はA・Bで選択したWorkを自動的に調べます。
# 固定データ内の別作品を探したい場合だけ入力してください。
C_TITLE = "" # @param {type:"string"}
C_AUTHOR = "" # @param {type:"string"}

if C_TITLE.strip() or C_AUTHOR.strip():
    q = df.copy()

    if C_TITLE.strip():
        q = q[
            q["ol_title"].str.contains(
                C_TITLE.strip(),
                case=False, regex=False, na=False
            )
        ]

    if C_AUTHOR.strip():
        q = q[
            q["ol_author_name"].str.contains(
                C_AUTHOR.strip(),
                case=False, regex=False, na=False
            )
        ]

    print("固定データ内のタイトル・著者検索")
else:
    # Work IDで正確に接続する。
    # 補正後のOL Work IDにも対応する。
    q = df[
        df["ol_work_id"].eq(wid)
        | df["current_ol_work_id"].eq(wid)
    ].copy()

    print("A・Bで選択したWork ID:", wid)

q = q.copy()

if not q.empty:
    q["_edition_sort"] = pd.to_numeric(
        q["ol_edition_count"],
        errors="coerce"
    ).fillna(-1)

    q = q.sort_values(
        "_edition_sort", ascending=False
    )

q["openalex_status_japanese"] = (
    q["openalex_quality_flag"]
    .map(FLAG_LABELS)
    .fillna("要確認")
)

print("固定データ内の該当OLレコード:", len(q))

if q.empty:
    print(
        "このWorkは34,789件の固定データに"
        "対応していません。"
    )
    print(
        "Open Library全体検索とEdition閲覧は"
        "引き続き利用できます。"
    )
else:
    display(q[[
        "ol_title",
        "ol_author_name",
        "ol_work_id",
        "current_ol_work_id",
        "ol_first_publish_year",
        "ol_edition_count",
        "wikidata_decision",
        "wikidata_qid",
        "openalex_display_candidate_count",
        "openalex_status_japanese",
    ]].head(30))

    print(
        "OpenAlex候補数は暫定値です。"
        "異なるOLレコードで繰り返される"
        "候補数を合算しないでください。"
    )


## C-2. 固定された作品集合を年代で見る
指定する年はOpen Libraryのソース上の最初の出版年で、厳密な文学史上の初出年とは限りません。このCSVは歴史的OL対象34,789レコードを収録しますが、すべての英語小説を網羅するものではありません。

In [ ]:
START_YEAR = 1880 # @param {type:"integer"}
END_YEAR = 1900 # @param {type:"integer"}
years = pd.to_numeric(df["ol_first_publish_year"], errors="coerce")
period = df[years.between(START_YEAR, END_YEAR)].copy()
print(f"この読み込みデータ内で{START_YEAR}–{END_YEAR}年に該当：{len(period):,} OLレコード")
display(period[["ol_title","ol_author_name","ol_first_publish_year"]].head(25))

## C-3. 登録Edition数を比較する
この列が入力CSVで欠損の場合は、数値が記録されていないことを示します。Editionは実際の刊行回数・刷数・売上・読者数ではありません。

In [ ]:
PLOT_TOP = 15 # @param {type:"integer", min:5, max:40}
x = period.copy()
x["edition_numeric"] = pd.to_numeric(x["ol_edition_count"], errors="coerce")
x = x.dropna(subset=["edition_numeric"]).nlargest(PLOT_TOP,"edition_numeric")
if x.empty: print("このデータにはEdition数が記録されていません。")
else:
    display(x[["ol_title","ol_author_name","ol_edition_count"]])
    ax=x.sort_values("edition_numeric").plot.barh(x="ol_title",y="edition_numeric",legend=False,figsize=(9,max(4,len(x)*.38)))
    ax.set(xlabel="Registered OL Edition records",ylabel="OL title"); plt.tight_layout(); plt.show()

## C-4. Wikidataとの照合状況
MATCHは採用された照合、AMBIGUOUSは曖昧、NO_MATCHは現行の探索・判定手続きでは照合できなかったことを意味します。

In [ ]:
d = df["wikidata_decision"].replace("", "NOT_RECORDED").value_counts(dropna=False)
display(d.to_frame("OL source records"))
d.plot.bar(figsize=(7,3),rot=0,title="Wikidata resolution status");plt.tight_layout();plt.show()

## C-5. OpenAlexの暫定候補数と注意フラグ

検索済みの作品について、OpenAlexの候補数とその状態を確認します。

- `PROVISIONAL`：Project Workに接続済みだが、候補数は暫定的。
- `PROVISIONAL_UNRESOLVED`：作品同定未確定。別の検索方法による参考値。
- `NOT_AVAILABLE`：表示できる候補数がない。0件という意味ではない。
- `IDENTITY_HOLD`：作品の同定に問題があり保留。

異なる方法で得た候補数を、そのまま比較・合算しないでください。


In [ ]:
print("=== 全体の状態フラグ ===")
display(
    df["openalex_quality_flag"]
    .value_counts()
    .rename_axis("状態")
    .reset_index(name="OLレコード数")
)

print("\n=== 検索した作品のOpenAlex候補 ===")
display(q[[
    "ol_title", "ol_work_id",
    "openalex_display_candidate_count",
    "openalex_quality_flag", "openalex_count_source",
    "openalex_count_unit_id", "openalex_qa_note",
]].head(30))

print("同じ計測単位IDに同じ候補数が繰り返される場合があります。")
print("同じ数値を複数作品分として加算しないでください。")


## C-6. 観察と解釈

- Edition数の多さは、読者が多かったことを意味するでしょうか。出版年・出版社・収録時点の偏りを考えてみましょう。
- WikidataでNO_MATCHだった作品は「存在しない作品」でしょうか。
- OpenAlex候補が多い作品は、学術的評価が高いと断定できるでしょうか。同名作品・一般語タイトル・タイトルと著者の誤結合が起こりえます。
- Open Library Workと研究上のProject Workは同一の概念でしょうか。今回の17件の作品同定保留事例を考えてみましょう。

### 発展：一作品のEditionを深く見る
この配布CSVは主にWorkごとのメタデータです。Editionの出版社・刊行年・言語を年表で扱うためには、事前に監査した別のEditionテーブルが必要です。未収集の詳細をこのCSVから補完したように表示しません。